# Sort, Validate, and Upload Raw Event Log Files

Takes an unsorted folder of raw event-log exports (e.g. `V1_4_1278_06.09.25.csv`, `MID_Short_Flywheel_1587_06.28.26.csv`), validates the `Subject` column inside each file against the subject encoded in its filename, **copies** the validated files into `ORGANIZED_DIR/<subject>/S1/<acquisition>/` (leaving `SOURCE_DIR` untouched), uploads them to the matching Flywheel acquisition (never overwriting an existing file), and — once every `func-bold*` acquisition on a session has an event log (excluding `SBRef` acquisitions) — marks `session.info['COMPLETENESS']['Stimulus Complete'] = True` while preserving every other key already in that object.

**Filename -> subject caveat:** for `V{1|2}_<run>_<subject>_<date>` files the subject is literally the 3rd `_`-delimited field, matching what was specified. `MID_Short_Flywheel_<subject>_<date>` files have a fixed 3-word prefix, so their subject is the 4th field (the 3rd field is the literal word `Flywheel`). `parse_filename()` below handles both correctly — flag if `MID_Short_Flywheel_*` files aren't actually meant to be included in this pass.

**Safety:** `DRY_RUN = True` by default. Both the Flywheel upload step and the `COMPLETENESS` metadata write are gated on it — review the printed output, then flip to `False` once it looks right.

## Setup

In [1]:
import shutil
import sys
from datetime import datetime
from pathlib import Path

import flywheel
import pandas as pd

SOURCE_DIR = Path('raw-data')  # <-- set this: unstructured raw exports, left untouched
ORGANIZED_DIR = Path('organized-data')  # <-- set this: where the sorted subject/session/acquisition tree is built
DRY_RUN = True  # gates the Flywheel upload, the COMPLETENESS write, and cleanup below

CONVERT_EPRIME_TXT = True  # set True to generate csvs from raw E-Prime .txt logs first
CONVERT_EPRIME_REPO_PATH = Path('convert-eprime')  # local clone of tsalo/convert-eprime

CLEANUP_AFTER_UPLOAD = False  # set True to delete local ORGANIZED_DIR copies once confirmed present on Flywheel

fw = flywheel.Client('')
project = fw.projects.find_one('label=YEARS,group=jgruber')
print('project found:', project.label)

project found: YEARS


## 0. (Optional) Convert raw E-Prime `.txt` logs to `.csv`

Only runs when `CONVERT_EPRIME_TXT = True`. Ported from the working conversion logic in `years-convert_files.ipynb`, using [convert-eprime](https://github.com/tsalo/convert-eprime)'s `text_to_csv`: reads each `.txt` as `utf-16`, converts it, then stamps an `ExperimentName` column (pulled from the `Experiment:` header line) onto the front and — for `YEARS` files — renames `Procedure` to `Procedure[Trial]`, exactly as the working notebook does.

**Output naming** matches the manually-exported convention `parse_filename()` expects (`V{1|2}_<run>_<subject>_<date>.csv` / `MID_Short_Flywheel_<subject>_<date>.csv`) rather than E-Prime's native `.txt` filename — the native name (e.g. `V1_1_YEARS-1909-1.txt`) doesn't parse correctly otherwise. The subject id is read from the converted file's own `Subject` column (authoritative) rather than parsed from the filename. The date comes from the file's own `SessionDate` field (`MM-DD-YYYY`, reformatted to the pipeline's `MM.DD.YY`) rather than today's date — falling back to today's date only if `SessionDate` is missing or unparseable. An `_autoconverted` suffix is appended so these are visibly distinguishable from manual exports, e.g. `V1_1_1909_08.12.26_autoconverted.csv`.

**Once a `.txt` is successfully converted (or its target `.csv` already exists), it's moved into `SOURCE_DIR/processed/`** — out of `SOURCE_DIR`'s top level — so step 2 never sees it (its filename pattern and non-csv/xlsx content would otherwise get it flagged there as a validation failure) and reruns of this step don't try to reconvert it. `.txt` files that fail to yield a usable `Subject` column are left in place at the top level for troubleshooting. Set `CONVERT_EPRIME_REPO_PATH` above to wherever you've cloned the repo.

In [2]:
def get_value_by_key(text, key):
    for line in text.splitlines():
        if line.startswith(f'{key}:'):
            return line.split(':', 1)[1].strip()
    return None


def normalize_subject_value(value):
    s = str(value).strip()
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except ValueError:
        pass
    return s


def get_session_date(df, txt_path):
    """Pull the session date from the file's own SessionDate field (MM-DD-YYYY),
    reformatted to the pipeline's MM.DD.YY convention. Falls back to today's date
    if SessionDate is missing/unparseable."""
    if 'SessionDate' in df.columns:
        raw_dates = df['SessionDate'].dropna().unique()
        if len(raw_dates) > 0:
            try:
                return datetime.strptime(str(raw_dates[0]), '%m-%d-%Y').strftime('%m.%d.%y')
            except ValueError:
                print(f'  could not parse SessionDate "{raw_dates[0]}" for {txt_path.name}, using today\'s date')

    print(f'  no usable SessionDate found for {txt_path.name}, using today\'s date')
    return datetime.now().strftime('%m.%d.%y')


def build_converted_filename(txt_path, subject, date_str):
    """Match the naming convention parse_filename() expects
    (V{1|2}_<run>_<subject>_<date>.csv or MID_Short_Flywheel_<subject>_<date>.csv)
    instead of E-Prime's native .txt filename, tagged with _autoconverted so
    these are visibly distinguishable from manual exports."""
    lower_stem = txt_path.stem.lower()

    if lower_stem.startswith(('v1_', 'v2_')):
        parts = txt_path.stem.split('_')
        version, run = parts[0], parts[1]
        base = f'{version}_{run}_{subject}'
    elif lower_stem.startswith('mid'):
        base = f'MID_Short_Flywheel_{subject}'
    else:
        base = txt_path.stem  # unrecognized native naming -- leave as-is rather than guess

    return f'{base}_{date_str}_autoconverted.csv'


if CONVERT_EPRIME_TXT:
    sys.path.insert(0, str(CONVERT_EPRIME_REPO_PATH))
    from convert_eprime.convert import text_to_csv

    processed_txt_dir = SOURCE_DIR / 'processed'
    processed_txt_dir.mkdir(parents=True, exist_ok=True)

    txt_files = sorted(SOURCE_DIR.glob('*.txt'))
    print(f'found {len(txt_files)} eprime .txt file(s) to convert')

    for txt_path in txt_files:
        tmp_csv_path = txt_path.with_suffix('.tmp.csv')

        with open(txt_path, 'r', encoding='utf-16') as f:
            text = f.read()

        text_to_csv(str(txt_path), str(tmp_csv_path))

        df = pd.read_csv(tmp_csv_path)
        df.insert(loc=0, column='ExperimentName', value=get_value_by_key(text, 'Experiment'))

        if 'YEARS' in txt_path.name:
            df = df.rename(columns={'Procedure': 'Procedure[Trial]'})

        if 'Subject' not in df.columns or df['Subject'].dropna().empty:
            print(f'no Subject column found in conversion of {txt_path.name}, leaving .txt in place for review')
            df.to_csv(tmp_csv_path, index=None)
            continue

        subject = normalize_subject_value(df['Subject'].dropna().iloc[0])
        date_str = get_session_date(df, txt_path)
        csv_path = txt_path.parent / build_converted_filename(txt_path, subject, date_str)

        if csv_path.exists():
            print(f'skip (csv already exists): {csv_path.name}')
            tmp_csv_path.unlink(missing_ok=True)
        else:
            df.to_csv(csv_path, index=None)
            tmp_csv_path.unlink(missing_ok=True)
            print(f'converted: {txt_path.name} -> {csv_path.name}')

        processed_txt_path = processed_txt_dir / txt_path.name
        shutil.move(str(txt_path), str(processed_txt_path))
        print(f'  moved {txt_path.name} -> processed/{processed_txt_path.name}')
else:
    print('CONVERT_EPRIME_TXT is False, skipping eprime .txt conversion step')

found 5 eprime .txt file(s) to convert
Output file successfully created- raw-data/MID_short-1909-1.tmp.csv
skip (csv already exists): MID_Short_Flywheel_1909_08.12.26_autoconverted.csv
  moved MID_short-1909-1.txt -> processed/MID_short-1909-1.txt
Output file successfully created- raw-data/V1_1_YEARS-1909-1.tmp.csv
converted: V1_1_YEARS-1909-1.txt -> V1_1_1909_08.12.26_autoconverted.csv
  moved V1_1_YEARS-1909-1.txt -> processed/V1_1_YEARS-1909-1.txt
Output file successfully created- raw-data/V1_2_YEARS-1909-1.tmp.csv
converted: V1_2_YEARS-1909-1.txt -> V1_2_1909_08.12.26_autoconverted.csv
  moved V1_2_YEARS-1909-1.txt -> processed/V1_2_YEARS-1909-1.txt
Output file successfully created- raw-data/V1_3_YEARS-1909-1.tmp.csv
converted: V1_3_YEARS-1909-1.txt -> V1_3_1909_08.12.26_autoconverted.csv
  moved V1_3_YEARS-1909-1.txt -> processed/V1_3_YEARS-1909-1.txt
Output file successfully created- raw-data/V1_4_YEARS-1909-1.tmp.csv
converted: V1_4_YEARS-1909-1.txt -> V1_4_1909_08.12.26_autocon

## 1. Parse subject / session / acquisition from filename

- `V{1|2}_<run>_<subject>_<date>.ext` -> `func-bold_task-years_dir-ap_run-<run, zero-padded>`
- `MID_Short_Flywheel_<subject>_<date>.ext` -> `func-bold_task-mid_dir-ap_run-01`
- Session is always `S1`.
- Anything else is left unrecognized and flagged for manual review rather than guessed at.

In [3]:
def parse_filename(filename):
    """Return (subject, session_label, acquisition_label) or None if unrecognized."""
    session_label = 'S1'

    if filename.startswith('MID_Short_Flywheel_'):
        # MID(1)_Short(2)_Flywheel(3)_<subject>(4)_<date>(5...)
        parts = filename.split('_')
        subject = parts[3]
        acquisition = 'func-bold_task-mid_dir-ap_run-01'
        return subject, session_label, acquisition

    if filename.startswith(('V1_', 'V2_')):
        # V{1|2}(1)_<run>(2)_<subject>(3)_<date>(4...) -- subject is the 3rd underscore-delimited field
        parts = filename.split('_')
        run_number = parts[1]
        subject = parts[2]
        run = f'{int(run_number):02d}'
        acquisition = f'func-bold_task-years_dir-ap_run-{run}'
        return subject, session_label, acquisition

    return None

## 2. Validate `Subject` column against filename, then copy into the organized tree

Reads each file's `Subject` column, normalizes numeric-looking ids (pandas often reads an int column as e.g. `1278.0`), and requires a single, exact match against the subject parsed from the filename. Files that don't match, have an unreadable/missing `Subject` column, or don't match either naming pattern are reported in `flagged` and left alone. Validated files are **copied** (original left in place in `SOURCE_DIR`) into `ORGANIZED_DIR/<subject>/S1/<acquisition>/<original filename>`.

In [4]:
def normalize_id(value):
    s = str(value).strip()
    if s.endswith('.0'):
        try:
            s = str(int(float(s)))
        except ValueError:
            pass
    return s


def find_csv_header_row(path, key_column, encoding='utf-8-sig'):
    """Some exports (e.g. convert_eprime with mismatched LogFrame counts) prepend a
    placeholder header + metadata line before the real header row. Scan for the row
    that actually contains key_column rather than assuming row 0."""
    with open(path, 'r', encoding=encoding, errors='replace') as f:
        for i, line in enumerate(f):
            fields = [c.strip() for c in line.rstrip('\n').split(',')]
            if key_column in fields:
                return i
    return None


def load_subject_values(path):
    key_column = 'Subject'

    if path.suffix.lower() == '.csv':
        header_row = find_csv_header_row(path, key_column)
        if header_row is None:
            raise ValueError(f"no '{key_column}' column found")
        df = pd.read_csv(path, skiprows=header_row, encoding='utf-8-sig')
    else:
        raw = pd.read_excel(path, header=None)
        header_row = next(
            (i for i, row in raw.iterrows() if key_column in row.astype(str).str.strip().values),
            None,
        )
        if header_row is None:
            raise ValueError(f"no '{key_column}' column found")
        df = pd.read_excel(path, header=header_row)

    if key_column not in df.columns:
        raise ValueError(f"no '{key_column}' column found")
    return sorted({normalize_id(v) for v in df[key_column].dropna()})


organized = []
flagged = []

for path in sorted(SOURCE_DIR.iterdir()):
    if not path.is_file() or path.name.startswith('.'):
        continue

    parsed = parse_filename(path.name)
    if parsed is None:
        flagged.append({'filename': path.name, 'reason': 'unrecognized filename pattern'})
        continue

    subject_from_name, session_label, acquisition_label = parsed

    try:
        subject_values = load_subject_values(path)
    except Exception as e:
        flagged.append({'filename': path.name, 'reason': f'could not read Subject column: {e}'})
        continue

    if len(subject_values) != 1:
        flagged.append({
            'filename': path.name,
            'reason': f'expected exactly one Subject value in file, found {subject_values}',
        })
        continue

    subject_in_file = subject_values[0]
    if subject_in_file != normalize_id(subject_from_name):
        flagged.append({
            'filename': path.name,
            'reason': f'Subject in file ("{subject_in_file}") does not match subject in filename ("{subject_from_name}")',
        })
        continue

    dest_dir = ORGANIZED_DIR / subject_from_name / session_label / acquisition_label
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_path = dest_dir / path.name
    shutil.copy2(str(path), str(dest_path))

    organized.append({
        'local_path': dest_path,
        'subject': subject_from_name,
        'session': session_label,
        'acquisition': acquisition_label,
        'filename': path.name,
    })

print(f'organized {len(organized)} file(s), flagged {len(flagged)} file(s) for review')
for item in flagged:
    print(f"  FLAGGED: {item['filename']} - {item['reason']}")

organized 5 file(s), flagged 0 file(s) for review


## 3. Upload organized files to Flywheel (never overwrite)

Scans the organized tree (`ORGANIZED_DIR/<subject>/S1/<acquisition>/<file>`) directly rather than relying on step 2's in-memory `organized` list, so this step is idempotent — safe to re-run even when step 2 found nothing new to copy (e.g. a prior run already organized everything). For each file found, resolves the matching subject/session/acquisition on Flywheel. If the acquisition already has a file with that exact name, it's skipped rather than overwritten. Missing subject/session/acquisition on Flywheel are reported as errors rather than auto-created.

In [5]:
def find_subjects(project, subject_id):
    """Query subjects by label. Numeric-looking labels (e.g. '1587') must be
    quoted in the filter string or Flywheel's query parser treats them as a
    number/regex and silently matches nothing against the string field."""
    subject_id = str(subject_id).strip()
    matches = {s.id: s for s in project.subjects.find(f'label="{subject_id}"')}
    return list(matches.values())


def iter_organized_files(organized_dir):
    """Walk ORGANIZED_DIR/<subject>/<session>/<acquisition>/<file>, regardless of
    whether the file was copied there by this run or a previous one."""
    for path in sorted(organized_dir.glob('*/*/*/*')):
        if not path.is_file():
            continue
        yield {
            'local_path': path,
            'subject': path.parent.parent.parent.name,
            'session': path.parent.parent.name,
            'acquisition': path.parent.name,
            'filename': path.name,
        }


upload_candidates = list(iter_organized_files(ORGANIZED_DIR))
print(f'found {len(upload_candidates)} organized file(s) to check against Flywheel')

uploaded = []
upload_skipped = []
upload_errors = []

for item in upload_candidates:
    subject_matches = find_subjects(project, item['subject'])

    if len(subject_matches) == 0:
        upload_errors.append({**item, 'reason': 'subject not found on Flywheel'})
        continue
    if len(subject_matches) > 1:
        ids = [s.id for s in subject_matches]
        upload_errors.append({
            **item,
            'reason': f'{len(subject_matches)} subjects match "{item["subject"]}" (ids: {ids}) - ambiguous, needs manual resolution',
        })
        continue

    subject = subject_matches[0]

    session = subject.sessions.find_one(f"label={item['session']}")
    if session is None:
        upload_errors.append({**item, 'reason': 'session not found on Flywheel'})
        continue

    acquisition = session.acquisitions.find_one(f"label={item['acquisition']}")
    if acquisition is None:
        upload_errors.append({**item, 'reason': 'acquisition not found on Flywheel'})
        continue

    acquisition = acquisition.reload()
    existing_names = {f.name for f in acquisition.files}

    label_path = f"{item['subject']}/{item['session']}/{item['acquisition']}/{item['filename']}"

    if item['filename'] in existing_names:
        upload_skipped.append(item)
        print(f'skip (already exists): {label_path}')
        continue

    if DRY_RUN:
        print(f'[dry run] would upload: {label_path}')
    else:
        acquisition.upload_file(str(item['local_path']))
        print(f'uploaded: {label_path}')

    uploaded.append(item)

print(f"\n{len(uploaded)} file(s) uploaded/would-upload, {len(upload_skipped)} skipped (already present), "
      f"{len(upload_errors)} error(s)")

found 10 organized file(s) to check against Flywheel
skip (already exists): 1587/S1/func-bold_task-mid_dir-ap_run-01/MID_Short_Flywheel_1587_06.28.26.csv
skip (already exists): 1587/S1/func-bold_task-years_dir-ap_run-01/V1_1_1587_06.28.26.csv
skip (already exists): 1587/S1/func-bold_task-years_dir-ap_run-02/V1_2_1587_06.28.26.csv
skip (already exists): 1587/S1/func-bold_task-years_dir-ap_run-03/V1_3_1587_06.28.26.csv
skip (already exists): 1587/S1/func-bold_task-years_dir-ap_run-04/V1_4_1587_06.28.26.csv
[dry run] would upload: 1909/S1/func-bold_task-mid_dir-ap_run-01/MID_Short_Flywheel_1909_08.12.26_autoconverted.csv
[dry run] would upload: 1909/S1/func-bold_task-years_dir-ap_run-01/V1_1_1909_08.12.26_autoconverted.csv
[dry run] would upload: 1909/S1/func-bold_task-years_dir-ap_run-02/V1_2_1909_08.12.26_autoconverted.csv
[dry run] would upload: 1909/S1/func-bold_task-years_dir-ap_run-03/V1_3_1909_08.12.26_autoconverted.csv
[dry run] would upload: 1909/S1/func-bold_task-years_dir-ap_ru

## 3b. (Optional) Clean up local organized copies after successful upload

Only runs when `CLEANUP_AFTER_UPLOAD = True`, and only deletes files from `uploaded` and `upload_skipped` — i.e. files *confirmed* present on Flywheel (either just uploaded, or already there from a previous run). Files in `upload_errors` (missing subject/session/acquisition, ambiguous subject match) are left alone so they're still available for troubleshooting. `SOURCE_DIR` is never touched by this step. Skipped automatically while `DRY_RUN = True`, since nothing was actually confirmed uploaded in that case.

In [6]:
cleaned_up = []

if CLEANUP_AFTER_UPLOAD and DRY_RUN:
    print('CLEANUP_AFTER_UPLOAD is True but DRY_RUN is also True -- skipping cleanup '
          '(nothing was actually confirmed uploaded this run)')
elif CLEANUP_AFTER_UPLOAD:
    for item in uploaded + upload_skipped:
        local_path = item['local_path']
        local_path.unlink(missing_ok=True)
        cleaned_up.append(item)

        # prune now-empty parent directories, but never remove ORGANIZED_DIR itself
        parent = local_path.parent
        while parent != ORGANIZED_DIR and parent.exists() and not any(parent.iterdir()):
            parent.rmdir()
            parent = parent.parent

    print(f'removed {len(cleaned_up)} local file(s) from {ORGANIZED_DIR} (confirmed present on Flywheel)')
else:
    print('CLEANUP_AFTER_UPLOAD is False, leaving organized copies in place')

CLEANUP_AFTER_UPLOAD is False, leaving organized copies in place


## 4. Mark `Stimulus Complete` once every func-bold acquisition has an event log

For each subject/session touched in this run, looks at **every** acquisition on that session whose label starts with `func-bold` (excluding ones ending in `SBRef`) — not just the ones uploaded to just now — and checks whether each already carries an event log (a raw file matching the naming patterns handled above, or a previously-curated file with `event` in its name). If all of them do, `session.info['COMPLETENESS']` is updated with `'Stimulus Complete': True`, reading the existing object first so none of its other keys (`Analysis ID`, `T1 count`, etc.) get clobbered.

In [7]:
def has_event_log(acquisition):
    for f in acquisition.files:
        name_lower = f.name.lower()
        if 'event' in name_lower or name_lower.startswith(('mid_short_flywheel', 'v1_', 'v2_')):
            return True
    return False


def get_func_bold_acquisitions(session):
    return [
        a.reload() for a in session.acquisitions.iter()
        if a.label.startswith('func-bold') and not a.label.endswith('SBRef')
    ]


touched_sessions = sorted({(item['subject'], item['session']) for item in upload_candidates})
completeness_updated = []
completeness_already_complete = []
completeness_errors = []

for subject_label, session_label in touched_sessions:
    subject_matches = find_subjects(project, subject_label)
    if len(subject_matches) != 1:
        completeness_errors.append((subject_label, session_label, f'{len(subject_matches)} matching subject(s)'))
        continue
    subject = subject_matches[0]

    session = subject.sessions.find_one(f'label={session_label}')
    if session is None:
        continue

    session = session.reload()
    func_bold_acqs = get_func_bold_acquisitions(session)

    if not func_bold_acqs or not all(has_event_log(a) for a in func_bold_acqs):
        continue

    completeness = dict(session.info.get('COMPLETENESS', {}))
    if completeness.get('Stimulus Complete') is True:
        print(f'already marked Stimulus Complete, nothing to do: {subject_label}/{session_label}')
        completeness_already_complete.append((subject_label, session_label))
        continue

    completeness['Stimulus Complete'] = True

    if DRY_RUN:
        print(f'[dry run] would mark Stimulus Complete: {subject_label}/{session_label}')
    else:
        session.update_info({'COMPLETENESS': completeness})
        print(f'marked Stimulus Complete: {subject_label}/{session_label}')

    completeness_updated.append((subject_label, session_label))

print(f'\n{len(completeness_updated)} session(s) marked Stimulus Complete (or would be, in dry run).')
print(f'{len(completeness_already_complete)} session(s) already had Stimulus Complete set.')
if completeness_errors:
    print('\nSkipped due to ambiguous/missing subject:')
    for subject_label, session_label, reason in completeness_errors:
        print(f'  {subject_label}/{session_label}: {reason}')

already marked Stimulus Complete, nothing to do: 1587/S1

0 session(s) marked Stimulus Complete (or would be, in dry run).
1 session(s) already had Stimulus Complete set.


## 5. Summary

In [8]:
print('=== Summary ===')
print(f'Organized (copied this run):    {len(organized)}')
print(f'Flagged (validation issues):    {len(flagged)}')
print(f'Uploaded (or dry-run preview):  {len(uploaded)}')
print(f'Skipped (already present):      {len(upload_skipped)}')
print(f'Upload errors (not on Flywheel):{len(upload_errors)}')
print(f'Local copies cleaned up:        {len(cleaned_up)}')
print(f'Sessions marked Stimulus Complete:      {len(completeness_updated)}')
print(f'Sessions already Stimulus Complete:     {len(completeness_already_complete)}')

if flagged:
    print('\nFiles needing manual review:')
    for item in flagged:
        print(f"  {item['filename']}: {item['reason']}")

if upload_errors:
    print('\nUpload errors:')
    for item in upload_errors:
        print(f"  {item['subject']}/{item['session']}/{item['acquisition']}/{item['filename']}: {item['reason']}")

=== Summary ===
Organized (copied this run):    5
Flagged (validation issues):    0
Uploaded (or dry-run preview):  5
Skipped (already present):      5
Upload errors (not on Flywheel):0
Local copies cleaned up:        0
Sessions marked Stimulus Complete:      0
Sessions already Stimulus Complete:     1
